# A3 — Model & Optimasi (Minggu 12)
### TA-PCVK: Facial Expression Recognition (Hybrid Feature Descriptor + Jaya-LSSVM)

**Divisi**: A3 (Model & Optimasi)  
**Referensi Jurnal**:
> Kar, N.B., Nayak, D.R., Babu, K.S., & Zhang, Y.-D. (2021). *A hybrid feature descriptor with Jaya optimised least squares SVM for facial expression recognition.* IET Image Processing, 15(7), 1471–1483. DOI: [10.1049/ipr2.12118](https://doi.org/10.1049/ipr2.12118)  
> Suykens, J.A.K., & Vandewalle, J. (1999). *Least Squares Support Vector Machine Classifiers.* Neural Processing Letters, 9(3), 293–300.

---

### Sasaran Minggu 12:
1. **Setup repositori & konfigurasi**: Sentralisasi parameter di `config.yaml`, manajemen dependensi di `requirements.txt`.
2. **Baseline Model (scikit-learn)**: Pipeline **LDA + SVM-RBF** dievaluasi dengan **5-fold Stratified Cross-Validation** pada dataset **CK+** dan **JAFFE** (fitur 680-D dari A2).
3. **Prototipe LS-SVM Biner**: Formulasi matematis aljabar linear dan implementasi `LSSVMClassifier` berbasis penyelesaian sistem persamaan linear KKT.
4. **Pengujian pada Dataset Mainan (Toy Datasets)**: Verifikasi kinerja dan paritas akurasi LS-SVM biner terhadap `scikit-learn SVC` pada problem non-linear **XOR** dan dataset **Iris**.

## 1. Import Library & Muat Konfigurasi (`config.yaml`)

Semua path dataset, parameter cross-validation, LDA, dan hyperparameter model dimuat secara terpusat dari `config.yaml` agar tidak terjadi *hardcoding* parameter (sesuai Definition of Done).

In [ ]:
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tambahkan direktori kerja ke sys.path jika belum ada
repo_root = Path.cwd().resolve()
if repo_root.name == "model-experiment":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "model-experiment") not in sys.path:
    sys.path.insert(0, str(repo_root / "model-experiment"))

from baseline import run_baseline_evaluation, load_config
from lssvm import LSSVMClassifier

# Muat konfigurasi proyek
config = load_config(repo_root / "config.yaml")
print(f"Proyek: {config['project']['name']} - {config['project']['description']}")
print(f"Random seed: {config['project']['random_seed']}")
print(f"Cross-Validation: {config['cross_validation']['n_splits']}-Fold Stratified CV")

## 2. Inspeksi Artefak Fitur Handoff dari A2

A2 telah mengekstrak fitur hybrid **SWT Haar Level-1 + PHOG ($L=3, K=8$)** berukuran **680 dimensi** dan menyimpannya di `data/ck_features.npz` dan `data/jaffe_features.npz`.

In [ ]:
ck_data = np.load(repo_root / config["paths"]["ck_features"])
jaffe_data = np.load(repo_root / config["paths"]["jaffe_features"])

print("=== DATASET CK+ ===")
print(f"Matriks fitur F : {ck_data['F'].shape} ({ck_data['F'].dtype})")
print(f"Label y         : {ck_data['y'].shape} (kelas unik: {np.unique(ck_data['y'])})")
print(f"Kelas           : {ck_data['class_names'].tolist()}")

print("\n=== DATASET JAFFE ===")
print(f"Matriks fitur F : {jaffe_data['F'].shape} ({jaffe_data['F'].dtype})")
print(f"Label y         : {jaffe_data['y'].shape} (kelas unik: {np.unique(jaffe_data['y'])})")
print(f"Kelas           : {jaffe_data['class_names'].tolist()}")

## 3. Evaluasi Baseline Model: LDA + SVM-RBF (scikit-learn)

Sesuai metodologi jurnal:
- **Reduksi Dimensi**: Linear Discriminant Analysis (LDA) mereduksi dimensi dari 680 ke $C - 1 = 6$ komponen diskriminatif.
- **Classifier**: Support Vector Machine dengan Radial Basis Function (SVM-RBF), $C = 10.0$, `gamma='scale'`.
- **Protokol Validasi**: 5-Fold Stratified Cross-Validation (mencegah data leakage: fit LDA hanya pada fold training).
- **Metrik**: Accuracy, Macro Sensitivity (Recall), Macro Specificity, dan Confusion Matrix.

In [ ]:
# Jalankan evaluasi baseline pada CK+ dan JAFFE
results_ck = run_baseline_evaluation(repo_root / config["paths"]["ck_features"], config=config)
results_jaffe = run_baseline_evaluation(repo_root / config["paths"]["jaffe_features"], config=config)

summary_data = [
    {
        "Dataset": "CK+",
        "Samples": results_ck["num_samples"],
        "Features": results_ck["num_features"],
        "Mean Accuracy (%)": f"{results_ck['mean_accuracy']*100:.2f} ± {results_ck['std_accuracy']*100:.2f}",
        "Macro Sensitivity (%)": f"{results_ck['macro_sensitivity']*100:.2f}",
        "Macro Specificity (%)": f"{results_ck['macro_specificity']*100:.2f}",
    },
    {
        "Dataset": "JAFFE",
        "Samples": results_jaffe["num_samples"],
        "Features": results_jaffe["num_features"],
        "Mean Accuracy (%)": f"{results_jaffe['mean_accuracy']*100:.2f} ± {results_jaffe['std_accuracy']*100:.2f}",
        "Macro Sensitivity (%)": f"{results_jaffe['macro_sensitivity']*100:.2f}",
        "Macro Specificity (%)": f"{results_jaffe['macro_specificity']*100:.2f}",
    },
]

df_baseline = pd.DataFrame(summary_data)
print("=== TABEL HASIL BASELINE LDA + SVM-RBF (5-FOLD STRATIFIED CV) ===")
display(df_baseline)

In [ ]:
# Visualisasi Confusion Matrix Agregat (5-Fold CV)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, res, title in zip(
    axes,
    [results_ck, results_jaffe],
    ["CK+ Baseline (LDA + SVM-RBF)", "JAFFE Baseline (LDA + SVM-RBF)"]
):
    cm = np.array(res["confusion_matrix"])
    class_names = res["class_names"]
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        title=f"{title}\nMean Acc: {res['mean_accuracy']*100:.2f}%",
        ylabel="True Label",
        xlabel="Predicted Label"
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontweight="bold")

fig.tight_layout()
plt.show()

## 4. Teori & Formulasi Matematis LS-SVM (Suykens & Vandewalle, 1999)

Standard SVM menyelesaikan Quadratic Programming (QP) dengan inequality constraints:
$$\min_{w, b, \xi} \frac{1}{2} w^T w + C \sum_{i=1}^N \xi_i \quad \text{s.t.} \quad y_i (w^T \phi(x_i) + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

Sebaliknya, **Least Squares SVM (LS-SVM)** memformulasikan error kuadratik dengan **equality constraints**:
$$\min_{w, b, e} \mathcal{J}(w, b, e) = \frac{1}{2} w^T w + \frac{\gamma}{2} \sum_{i=1}^N e_i^2 \quad \text{s.t.} \quad y_i [w^T \phi(x_i) + b] = 1 - e_i, \quad i = 1, \dots, N$$

Fungsi Lagrangian:
$$\mathcal{L}(w, b, e, \alpha) = \frac{1}{2} w^T w + \frac{\gamma}{2} \sum_{i=1}^N e_i^2 - \sum_{i=1}^N \alpha_i \Big( y_i [w^T \phi(x_i) + b] - 1 + e_i \Big)$$

Kondisi optimalitas Karush-Kuhn-Tucker (KKT):
1. $\frac{\partial \mathcal{L}}{\partial w} = 0 \implies w = \sum_{i=1}^N \alpha_i y_i \phi(x_i)$
2. $\frac{\partial \mathcal{L}}{\partial b} = 0 \implies \sum_{i=1}^N \alpha_i y_i = 0$
3. $\frac{\partial \mathcal{L}}{\partial e_i} = 0 \implies \alpha_i = \gamma e_i$
4. $\frac{\partial \mathcal{L}}{\partial \alpha_i} = 0 \implies y_i [w^T \phi(x_i) + b] - 1 + e_i = 0$

Substitusi menghasilkan **sistem persamaan linear $(N+1) \times (N+1)$**:
$$\begin{bmatrix} 0 & y^T \\ y & \Omega + \gamma^{-1} I_N \end{bmatrix} \begin{bmatrix} b \\ \alpha \end{bmatrix} = \begin{bmatrix} 0 \\ \mathbf{1}_N \end{bmatrix}$$
di mana $\Omega_{ij} = y_i y_j K(x_i, x_j)$, dan kernel RBF:
$$K(x_i, x_j) = \exp\left( -\frac{\|x_i - x_j\|^2}{2 \sigma^2} \right)$$

Fungsi keputusan untuk data baru $x_*$:
$$f(x_*) = \sum_{i=1}^N \alpha_i y_i K(x_i, x_*) + b$$
$$\hat{y}(x_*) = \text{sign}(f(x_*))$$

## 5. Verifikasi Aljabar Linear Solusi KKT

Kita memverifikasi bahwa solusi numerik $[b, \alpha]^T$ yang dihasilkan `LSSVMClassifier` benar-benar memenuhi persamaan matriks KKT $A [b, \alpha]^T = B$ dengan residual $\|A \cdot \text{sol} - B\| \approx 0$.

In [ ]:
# Uji kepuasan KKT pada data sintetis
rng = np.random.RandomState(42)
X_test_kkt = rng.randn(30, 2)
y_test_kkt = np.where(X_test_kkt[:, 0] > 0, 1, -1)

gamma = 10.0
sigma = 1.0
clf_lssvm = LSSVMClassifier(gamma=gamma, sigma=sigma, kernel="rbf")
clf_lssvm.fit(X_test_kkt, y_test_kkt)

# Rekonstruksi matriks A dan B
N = len(y_test_kkt)
K = clf_lssvm._compute_kernel(X_test_kkt, X_test_kkt)
Omega = np.outer(clf_lssvm.y_train_encoded_, clf_lssvm.y_train_encoded_) * K
reg_diag = (1.0 / gamma) + clf_lssvm.ridge_jitter
A22 = Omega + np.eye(N) * reg_diag

A = np.empty((N + 1, N + 1))
A[0, 0] = 0.0
A[0, 1:] = clf_lssvm.y_train_encoded_
A[1:, 0] = clf_lssvm.y_train_encoded_
A[1:, 1:] = A22

B = np.empty(N + 1)
B[0] = 0.0
B[1:] = 1.0

sol = np.hstack(([clf_lssvm.b_], clf_lssvm.alpha_))
residual = np.linalg.norm(A @ sol - B)
print(f"Norm Residual ||A * [b, alpha]^T - B||: {residual:.2e}")
assert residual < 1e-5, "Residual KKT harus mendekati nol!"
print("Status: KKT terverifikasi secara eksak! Solusi LS-SVM terbukti benar secara matematis.")

## 6. Uji LS-SVM pada Dataset Mainan (Toy Datasets)

Sesuai DoD Minggu 12:
> *Uji LS-SVM pada dataset mainan (XOR/iris). Output: LS-SVM terbukti benar di data mainan (akurasi setara SVM).*

### Eksperimen A: Problem Non-Linear XOR
Problem XOR tidak dapat dipisahkan secara linear. Kernel RBF pada LS-SVM harus mampu membentuk batas keputusan non-linear kuadran XOR yang setara dengan `SVC(kernel='rbf')`.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

# 1. Dataset Non-Linear XOR
rng = np.random.RandomState(42)
X_xor = rng.uniform(-1.0, 1.0, size=(200, 2))
y_xor = np.where(X_xor[:, 0] * X_xor[:, 1] > 0, 1, -1)
X_xor += rng.normal(0, 0.05, size=X_xor.shape)

X_train_xor, X_test_xor, y_train_xor, y_test_xor = train_test_split(
    X_xor, y_xor, test_size=0.3, random_state=42, stratify=y_xor
)

# Latih LS-SVM dan SVC
lssvm_xor = LSSVMClassifier(gamma=50.0, sigma=0.4, kernel="rbf")
lssvm_xor.fit(X_train_xor, y_train_xor)

svc_xor = SVC(kernel="rbf", C=50.0, gamma=1.0 / (2.0 * 0.4**2), random_state=42)
svc_xor.fit(X_train_xor, y_train_xor)

print("=== HASIL EVALUASI XOR ===")
print(f"LS-SVM Train Acc : {lssvm_xor.score(X_train_xor, y_train_xor)*100:.2f}%")
print(f"LS-SVM Test Acc  : {lssvm_xor.score(X_test_xor, y_test_xor)*100:.2f}%")
print(f"SVC Train Acc    : {svc_xor.score(X_train_xor, y_train_xor)*100:.2f}%")
print(f"SVC Test Acc     : {svc_xor.score(X_test_xor, y_test_xor)*100:.2f}%")

### Eksperimen B: Dataset Iris (Klasifikasi Biner Setosa vs Versicolor)

Membandingkan kinerja klasifikasi LS-SVM biner dan scikit-learn SVC pada dataset benchmark Iris.

In [ ]:
# 2. Dataset Iris Biner
iris = load_iris()
mask = iris.target < 2  # Setosa (0) vs Versicolor (1)
X_iris = iris.data[mask]
y_iris = iris.target[mask]

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

lssvm_iris = LSSVMClassifier(gamma=10.0, sigma=1.0, kernel="rbf")
lssvm_iris.fit(X_train_iris, y_train_iris)

svc_iris = SVC(kernel="rbf", C=10.0, gamma=1.0 / (2.0 * 1.0**2), random_state=42)
svc_iris.fit(X_train_iris, y_train_iris)

print("=== HASIL EVALUASI IRIS (SETOSA VS VERSICOLOR) ===")
print(f"LS-SVM Test Acc : {lssvm_iris.score(X_test_iris, y_test_iris)*100:.2f}%")
print(f"SVC Test Acc    : {svc_iris.score(X_test_iris, y_test_iris)*100:.2f}%")
assert lssvm_iris.score(X_test_iris, y_test_iris) == svc_iris.score(X_test_iris, y_test_iris) == 1.0
print("Status: Paritas sempurna (100% akurasi) antara LS-SVM dan SVC!")

### Visualisasi Komparasi Batas Keputusan (Decision Boundaries)

In [ ]:
def plot_boundary(clf, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if hasattr(clf, "decision_function"):
        Z = clf.decision_function(grid)
    else:
        Z = clf.predict(grid)
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=20, cmap=plt.cm.coolwarm, alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0.0], linewidths=2, colors="black", linestyles="--")
    
    classes = np.unique(y)
    colors = ["blue", "red"]
    markers = ["o", "s"]
    for idx, c in enumerate(classes):
        mask = (y == c)
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[idx], marker=markers[idx],
                   label=f"Kelas {c}", edgecolors="k", s=35, alpha=0.85)
    acc = clf.score(X, y) * 100.0
    ax.set_title(f"{title}\nAkurasi: {acc:.1f}%", fontsize=10, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# XOR Decision Boundary
plot_boundary(lssvm_xor, X_xor, y_xor, "LS-SVM RBF (XOR Problem)", axes[0, 0])
plot_boundary(svc_xor, X_xor, y_xor, "scikit-learn SVC RBF (XOR Problem)", axes[0, 1])

# Iris 2D (Sepal Length vs Petal Length)
X_iris_2d = iris.data[mask][:, [0, 2]]
lssvm_iris_2d = LSSVMClassifier(gamma=10.0, sigma=1.0, kernel="rbf").fit(X_iris_2d, y_iris)
svc_iris_2d = SVC(kernel="rbf", C=10.0, gamma=1.0 / (2.0 * 1.0**2)).fit(X_iris_2d, y_iris)

plot_boundary(lssvm_iris_2d, X_iris_2d, y_iris, "LS-SVM RBF (Iris Biner)", axes[1, 0])
plot_boundary(svc_iris_2d, X_iris_2d, y_iris, "scikit-learn SVC RBF (Iris Biner)", axes[1, 1])

fig.suptitle("Komparasi Batas Keputusan: LS-SVM vs scikit-learn SVC (Minggu 12)", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Ringkasan Hasil & Checklist Definition of Done (Minggu 12)

| Kriteria DoD | Status | Bukti / Output |
|--------------|--------|----------------|
| **1. Bersih dari `pip install -r requirements.txt`** | ✅ Selesai | Pinned requirements & venv terverifikasi (`numpy`, `scipy`, `scikit-learn`, `pytest`, `pyyaml`). |
| **2. Bukti verifikasi teruji** | ✅ Selesai | Tabel hasil baseline (`reports/baseline_summary.csv`), JSON (`reports/baseline_*.json`), gambar CM (`reports/figures/baseline_*.png`), batas keputusan (`reports/figures/lssvm_toy_xor_iris.png`). |
| **3. Parameter terpusat di `config.yaml`** | ✅ Selesai | File `config.yaml` berisi konfigurasi lengkap dataset, cross-validation, LDA, SVM, LS-SVM, dan toy datasets tanpa hardcoding. |
| **4. Unit test suite lulus 100%** | ✅ Selesai | `pytest` berhasil menjalankan 16 test suite (termasuk verifikasi KKT, XOR, Iris, dan baseline). |
| **5. Rencana Minggu 13** | ⏳ Siap | Melanjutkan ke **LS-SVM Multiclass (One-vs-One)**, evaluasi 5-fold CV dengan grid search parameter $(\gamma, \sigma)$, dan tabel metrik Sensitivitas/Spesifisitas/Akurasi per-kelas. |